# 1-board: ntc-curve

Grade the board NTC's reading against the thermocouple, and refit the curve the firmware converts with, so a board temperature in the log is a temperature.

## Purpose

The firmware derates and cuts off on the board NTC, and the winding thermometer leans on it as the case's temperature. It converts the raw count with a quadratic in the control table, seeded from the part's nominal beta. This experiment puts the thermocouple on the board next to the NTC, warms the board through the range the thermal experiments use, and compares the firmware's conversion and the part's beta equation with what the thermocouple reads.

## Procedure

The board on a controlled heat source with the thermocouple tip taped to it next to the NTC, on the DPS at 7.4 V with the bridge off. Any heater that holds a set temperature does, a 3D printer's heated bed, a hot plate with a thermostat, or a filament dryer, as long as the whole board sits at one temperature and only the board is on it. The DMM logs the thermocouple throughout. `capture.py` polls the NTC's raw count every two seconds, records the conversion constants the board holds, and marks the plateaus as the heater is set to about 45 C, then 60 C, then switched off.

1. The board, the thermocouple and the heater as above; the DMM on temperature, Bluetooth on.
2. `capture.py`. It asks at each plateau and marks it.

## Data

- `dmm.csv`: the thermocouple's log, `t_host`, `function`, `value` in C.
- `ntc.csv`: `t_host` and `ntc_raw`, the raw count every two seconds.
- `ntc-constants.json`: `ntc_raw_ref`, `ntc_t_ref_cc`, `ntc_k1_q88`, `ntc_k2_q24`, the quadratic the firmware converts with, read from the board.
- `marks.csv`: the host time of each plateau prompt.

### The two logs

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from oscnb import b41t, boards, display, laps, results, tel
from oscnb.display import show

BLUE, ORANGE, INK, MUTED = "#2a78d6", "#eb6834", "#0b0b0b", "#898781"
BOARD = boards.BOARDS[boards.DEFAULT]
CAPTURE = Path("../../../telemetry") / BOARD.key / "ntc-curve" / "capture-1"
CAPTURE = CAPTURE.resolve()
print(BOARD.key, "from", results.rel(CAPTURE))

In [ ]:
from oscnb import ntc as ntcmod
dmm = b41t.load(CAPTURE / "dmm.csv"); n = pd.read_csv(CAPTURE / "ntc.csv"); marks = pd.read_csv(CAPTURE / "marks.csv")
tc = results.read("0-instruments", "thermocouple"); lag = results.read("0-instruments", "dmm")["lag_s"]["value"]
dmm["t_c"] = (dmm["value"] - tc["offset_c"]["value"]) / tc["slope"]["value"]
dmm["t_true"] = dmm["t_host"] - lag
t0 = n["t_host"].iloc[0]
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(dmm["t_true"] - t0, dmm["t_c"], ".", ms=2, color=BLUE); ax[0].set_ylabel("thermocouple (C)"); ax[0].set_xlabel("time (s)"); ax[0].grid(alpha=0.3)
ax[1].plot(n["t_host"] - t0, n["ntc_raw"], ".", ms=2, color=ORANGE); ax[1].set_ylabel("ntc_raw (counts)"); ax[1].set_xlabel("time (s)"); ax[1].grid(alpha=0.3)
for a in ax:
    for _, m in marks.iterrows():
        a.axvline(m["t_host"] - t0, color=MUTED, lw=0.6); a.text(m["t_host"] - t0, a.get_ylim()[1], m["mark"], fontsize=8, va="top")
plt.show()
print(f"{len(dmm)} thermocouple packets, {len(n)} NTC polls over {(n['t_host'].iloc[-1] - t0) / 60:.0f} min")

The thermocouple, corrected by its stage 0 offset and slope and shifted by the DMM's lag, and the NTC's raw count, through the plateaus.

## Method

The thermocouple's reading is interpolated onto each NTC poll. The polls during the ramps are where the two sensors may lag each other, so the fit uses the whole run but the grading reads the plateaus. Three curves are compared with the thermocouple: the firmware's own conversion as logged, the part's beta equation at the nominal beta and R25, and the beta equation with its beta refitted. The quadratic the firmware carries is then refitted through the measured points, with the reference count kept, as the constants a new board entry would get.

### The NTC against the thermocouple

In [ ]:
n["t_c"] = np.interp(n["t_host"], dmm["t_true"], dmm["t_c"])
k = json.loads((CAPTURE / "ntc-constants.json").read_text())
n["fw_c"] = ntcmod.firmware_cc(n["ntc_raw"], k["ntc_raw_ref"], k["ntc_t_ref_cc"], k["ntc_k1_q88"], k["ntc_k2_q24"]) / 100
n["beta_c"] = BOARD.ntc_c_beta(n["ntc_raw"])
beta_fit = ntcmod.fit_beta(BOARD.ntc_ohm(n["ntc_raw"]), n["t_c"], BOARD.ntc_r25_ohm)
n["beta_fit_c"] = 1 / (1 / 298.15 + np.log(BOARD.ntc_ohm(n["ntc_raw"]) / BOARD.ntc_r25_ohm) / beta_fit) - 273.15
raw_ref = 2048
t_ref_cc, k1, k2 = ntcmod.fit_quadratic(n["ntc_raw"], n["t_c"], raw_ref)
n["quad_c"] = ntcmod.firmware_cc(n["ntc_raw"], raw_ref, t_ref_cc, k1, k2) / 100
err = {c: n[c] - n["t_c"] for c in ("fw_c", "beta_c", "beta_fit_c", "quad_c")}
for c, e in err.items():
    print(f"{c:<11} mean {e.mean():+.2f} C, worst {e.abs().max():.2f} C")
print(f"beta refit {beta_fit:.0f} against {BOARD.ntc_beta} nominal; quadratic through the points: t_ref {t_ref_cc} cC, k1 {k1} (Q8.8), k2 {k2} (Q24) at raw_ref {raw_ref}")
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(n["t_c"], n["ntc_raw"], ".", ms=2, color=BLUE); ax[0].set_xlabel("thermocouple (C)"); ax[0].set_ylabel("ntc_raw (counts)"); ax[0].grid(alpha=0.3)
for c, col in (("fw_c", INK), ("beta_c", MUTED), ("quad_c", ORANGE)):
    ax[1].plot(n["t_c"], err[c], ".", ms=2, color=col, label=c)
ax[1].axhline(0, color=INK, lw=0.5); ax[1].set_xlabel("thermocouple (C)"); ax[1].set_ylabel("conversion less thermocouple (C)"); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.show()

Left, the raw count against the thermocouple over the whole run. Right, each conversion's error against the thermocouple: the firmware's as logged, the nominal beta, and the quadratic refitted through the points.

## Results

The firmware conversion's error over the range, the refitted beta, and the quadratic's constants as a new entry would carry them.

In [ ]:
e = err["fw_c"]
results.write("1-board", "ntc-curve", CAPTURE, pair=BOARD.key,
              notes="board NTC polled while the board warmed and cooled on a controlled heat source, thermocouple taped next to it; t_c is the thermocouple after its stage 0 correction",
              firmware_error_c=results.quantity(e.mean(), lo=e.min(), hi=e.max(), unit="C"),
              beta_fit=results.quantity(beta_fit, unit="K"),
              ntc_raw_ref=results.quantity(raw_ref, unit="counts"), ntc_t_ref_cc=results.quantity(t_ref_cc, unit="cC"),
              ntc_k1_q88=results.quantity(k1, unit="Q8.8"), ntc_k2_q24=results.quantity(k2, unit="Q24"),
              range_c=results.quantity(n["t_c"].max(), lo=n["t_c"].min(), hi=n["t_c"].max(), unit="C"))

## Limitations

- The thermocouple is taped beside the NTC, not inside it. On a ramp the two lag each other, and the grading rests on the plateaus.
- The range is room to about 60 C, as far as the heater is taken. The cutoff temperatures above it are extrapolated.
- The thermocouple's own accuracy is the stage 0 result, and the DMM's temperature accuracy from the manual is still to be added there.

## Open questions

- How much self-heating the NTC's pull-up adds. The part sits at a milliwatt or less, which is under a tenth of a degree in still air, and the same run with the bridge driving the load would show whether the board's own heat moves it.

## Conclusion

To be written from the capture.